In [ ]:
%run ./_common

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
np.set_printoptions(precision=2)

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)

In [ ]:
SA = SWSAcademyHelper()
SA.init()

In [ ]:
airbnb_base_path = SA.datasets.airbnb.base_path
df = spark.read.format("csv").option("header", "true").load(f"{airbnb_base_path}/sf-listings/airbnb-cleaned-mlflow.csv")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SA.catalog_name}.{SA.schema_name}.airbnb_cleaned_mlflow")

In [ ]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SA.catalog_name}.{SA.schema_name}.ray_logs")
print(f"Ensured volume exists for Ray logs at {SA.catalog_name}.{SA.schema_name}.ray_logs")

In [ ]:
# Ensure a fresh volume for Ray logs
try:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {SA.catalog_name}.{SA.schema_name}.ray_logs")
    print(f"Ensured volume exists for Ray logs at {SA.catalog_name}.{SA.schema_name}.ray_logs")
except Exception as e:
    print("Warning: Could not create volume:", str(e))

In [ ]:
# Spark ML models logged or loaded with MLflow need a temporary folder. The default (/dbfs/tmp) is blocked when the
# workspace disables the DBFS root, so use a Unity Catalog volume in the learner schema instead.
import os
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SA.catalog_name}.{SA.schema_name}.mlflow_tmp")
os.environ["MLFLOW_DFS_TMP"] = f"/Volumes/{SA.catalog_name}/{SA.schema_name}/mlflow_tmp"

# On dedicated (single-user) clusters MLflow loads Spark ML models through the DBFS root (/dbfs/...), which is blocked
# when the workspace disables it. Use MLflow's own Unity Catalog volume loader instead.
import mlflow.spark as _mlflow_spark
if hasattr(_mlflow_spark, "_load_model_databricks_uc_volume"):
    _mlflow_spark._load_model_databricks_dbfs = _mlflow_spark._load_model_databricks_uc_volume

In [ ]:
# The Airbnb file has a few malformed rows (amenities text spilling into numeric columns). Databricks Runtime 17+
# turns on ANSI SQL mode by default, which stops on such values; the course was built for 16.4, where they become
# null. Use the 16.4 behaviour on any runtime.
try:
    spark.conf.set("spark.sql.ansi.enabled", "false")
except Exception:
    pass